# Exploratory Data Analysis: NYC short-term rentals

This notebook explores the raw rental listings dataset to understand its shape,
find data quality problems, and decide on the cleaning steps the pipeline will apply.
All analysis is tracked in Weights & Biases so the exploration is reproducible.

## Imports

Load the libraries used below: `wandb` for experiment tracking, `pandas` for data
manipulation, and `pandas_profiling` for automatic dataset profiling.


In [1]:
import wandb
import pandas as pd
import pandas_profiling


## Fetching the raw data from Weights & Biases

Start a tracked run and download the `sample.csv` artifact produced by the
`download` pipeline step. The CSV is read into a pandas DataFrame for analysis.


In [2]:
# Download Artifact from WB and read the csv file 
run = wandb.init(project="nyc_airbnb", group="eda", save_code=True)
local_path = wandb.use_artifact("sample.csv:latest").file()
df = pd.read_csv(local_path)

wandb: Currently logged in as: raminai (use `wandb login --relogin` to force relogin)
wandb: wandb version 0.12.11 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade


## Profiling the dataset

Generate an automatic profile report: column types, missing values, distributions,
and warnings. Notable findings: missing values in several columns, `last_review`
stored as strings instead of dates, and price outliers (zero and very high prices).


In [6]:
# # Perform Profiling on the DF
profile = pandas_profiling.ProfileReport(df)
# # profile.to_widgets()

## Viewing the profile report

Render the report inline. After reviewing it with stakeholders, the cleaning rules
for the pipeline were set: keep prices between $10 and $350 per night, and parse
`last_review` as a datetime. Missing values are deliberately left alone here:
they are imputed later in the inference pipeline so production data with gaps is
handled the same way.


In [5]:
profile

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

## Applying the first fixes

Apply the two fixes identified from the profile: drop price outliers outside the
$10 to $350 range, and convert `last_review` from strings to datetimes.


In [7]:
# Drop outliers
min_price = 10
max_price = 350
idx = df['price'].between(min_price, max_price)
df = df[idx].copy()
# Convert last_review to datetime
df['last_review'] = pd.to_datetime(df['last_review'])

## Verifying the fixes

Check the DataFrame structure after the fixes: confirm the row count dropped by
the expected outlier rows and that `last_review` is now a datetime column.


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 19001 entries, 0 to 19999
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   id                              19001 non-null  int64         
 1   name                            18994 non-null  object        
 2   host_id                         19001 non-null  int64         
 3   host_name                       18993 non-null  object        
 4   neighbourhood_group             19001 non-null  object        
 5   neighbourhood                   19001 non-null  object        
 6   latitude                        19001 non-null  float64       
 7   longitude                       19001 non-null  float64       
 8   room_type                       19001 non-null  object        
 9   price                           19001 non-null  int64         
 10  minimum_nights                  19001 non-null  int64         
 11  nu

## Closing the run

Finish the Weights & Biases run so the notebook and its results are versioned
alongside the data artifact used here.


In [9]:
run.finish()